# 8.1 量化 (Quantization)

量化通过降低模型参数的数值精度来减少模型大小和加速推理，是最实用的模型压缩技术。

本节涵盖：
- 训练后量化（PTQ）
- 量化感知训练（QAT）
- GPTQ
- AWQ
- SmoothQuant

## 1. 训练后量化（PTQ）

**目的**：无需重训练即可量化模型

**基本原理**：在模型训练完成后，将FP32/FP16权重直接映射到低精度（INT8/INT4），使用校准数据集确定量化参数。

**量化公式**：
- q = round(x / scale + zero_point)
- x_dequant = (q - zero_point) * scale
- scale = (x_max - x_min) / (q_max - q_min)

**对称量化 vs 非对称量化**：
- 对称：zero_point=0，适用于权重（近似对称分布）
- 非对称：zero_point≠0，适用于激活值（可能有偏移）

In [ ]:
import torch

torch.manual_seed(42)

def symmetric_quantize(tensor, n_bits=8):
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -(2 ** (n_bits - 1))
    max_val = tensor.abs().max()
    scale = max_val / q_max
    quantized = torch.clamp(torch.round(tensor / scale), q_min, q_max).to(torch.int8)
    dequantized = quantized.float() * scale
    return quantized, dequantized, scale

def asymmetric_quantize(tensor, n_bits=8):
    q_max = 2 ** n_bits - 1
    x_min, x_max = tensor.min(), tensor.max()
    scale = (x_max - x_min) / q_max
    zero_point = torch.round(-x_min / scale)
    quantized = torch.clamp(torch.round(tensor / scale + zero_point), 0, q_max).to(torch.uint8)
    dequantized = (quantized.float() - zero_point) * scale
    return quantized, dequantized, scale, zero_point

W = torch.randn(256, 256)

q_sym, dq_sym, scale_sym = symmetric_quantize(W, n_bits=8)
q_asym, dq_asym, scale_asym, zp_asym = asymmetric_quantize(W, n_bits=8)

print('=== Post-Training Quantization (PTQ) ===')
print(f'Original: FP32, {W.numel() * 4 / 1e6:.2f} MB')
print(f'INT8: {W.numel() / 1e6:.2f} MB (4x compression)')

sym_error = (W - dq_sym).abs().mean() / W.abs().mean()
asym_error = (W - dq_asym).abs().mean() / W.abs().mean()
print(f'\nSymmetric quantization relative error: {sym_error:.4f}')
print(f'Asymmetric quantization relative error: {asym_error:.4f}')
print(f'\nKey: PTQ is simple and fast, but may lose accuracy for very low bits.')

## 2. 量化感知训练（QAT）

**目的**：训练时模拟量化效果，获得更高质量的量化模型

**基本原理**：在前向传播中插入伪量化节点（fake quantization），模拟量化-反量化的精度损失，反向传播使用直通估计器（STE）绕过不可导的取整操作。

**STE（Straight-Through Estimator）**：
- 前向：x_q = round(x / s) * s（量化+反量化）
- 反向：∂L/∂x = ∂L/∂x_q（梯度直接传递，忽略取整操作）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class FakeQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, scale, q_min, q_max):
        x_q = torch.clamp(torch.round(x / scale), q_min, q_max)
        return x_q * scale

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None, None, None

class QATLinear(nn.Module):
    def __init__(self, in_features, out_features, n_bits=8):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.n_bits = n_bits
        self.q_max = 2 ** (n_bits - 1) - 1
        self.q_min = -(2 ** (n_bits - 1))
        self.weight_scale = None
        self.act_scale = None

    def forward(self, x):
        if self.weight_scale is None:
            self.weight_scale = self.linear.weight.abs().max() / self.q_max
        if self.act_scale is None:
            self.act_scale = x.abs().max() / self.q_max
        w_q = FakeQuantize.apply(self.linear.weight, self.weight_scale, self.q_min, self.q_max)
        x_q = FakeQuantize.apply(x, self.act_scale, 0, 2**self.n_bits - 1)
        return F.linear(x_q, w_q, self.linear.bias)

qat_layer = QATLinear(128, 64, n_bits=8)
x = torch.randn(4, 128)
out = qat_layer(x)
loss = out.sum()
loss.backward()

print('=== Quantization-Aware Training (QAT) ===')
print(f'Input: {x.shape}, Output: {out.shape}')
print(f'Weight scale: {qat_layer.weight_scale:.6f}')
print(f'Activation scale: {qat_layer.act_scale:.6f}')
print(f'Gradients flow through fake-quantize (STE)')
print(f'\nKey: QAT simulates quantization during training,')
print(f'resulting in better accuracy after actual quantization.')

## 3. GPTQ / AWQ / SmoothQuant

**GPTQ**：基于近似二阶信息的训练后量化方法，逐层量化权重，使用Hessian矩阵信息最小化量化误差。

**AWQ（Activation-Aware Weight Quantization）**：基于激活值重要性感知的权重量化，对重要权重通道（激活值大的通道）使用更高精度。

**SmoothQuant**：将激活值中的量化难度迁移到权重中，通过数学等价变换使激活值更易量化。

**对比**：
| 方法 | 精度 | 速度 | 适用场景 |
|------|------|------|----------|
| PTQ | 中 | 最快 | 快速部署 |
| GPTQ | 高 | 快 | 4-bit量化 |
| AWQ | 高 | 快 | 4-bit量化 |
| SmoothQuant | 高 | 快 | 8-bit W8A8 |
| QAT | 最高 | 慢 | 极致精度 |

In [ ]:
import torch

torch.manual_seed(42)

W = torch.randn(256, 512)
x = torch.randn(32, 256)

def awq_quantize(W, x, n_bits=4):
    with torch.no_grad():
        act_scales = x.abs().max(dim=0)[0]
    weight_scales = W.abs().max(dim=1)[0]
    importance = act_scales * weight_scales
    q_max = 2 ** (n_bits - 1) - 1
    scale = W.abs().max(dim=1, keepdim=True)[0] / q_max
    W_q = torch.clamp(torch.round(W / scale), -q_max, q_max)
    W_dq = W_q * scale
    return W_dq, importance

def smoothquant_transform(W, x, alpha=0.5):
    act_scales = x.abs().max(dim=0)[0]
    weight_scales = W.abs().max(dim=0)[0]
    smooth_scale = torch.pow(act_scales / (weight_scales + 1e-8), alpha)
    smooth_scale = torch.clamp(smooth_scale, min=1e-5)
    W_smooth = W * smooth_scale.unsqueeze(0)
    x_smooth = x / smooth_scale.unsqueeze(0)
    return W_smooth, x_smooth

W_awq, importance = awq_quantize(W, x, n_bits=4)
W_sq, x_sq = smoothquant_transform(W, x)

awq_error = (W - W_awq).norm() / W.norm()
orig_out = x @ W.T
sq_out = x_sq @ W_sq.T
sq_error = (orig_out - sq_out).norm() / orig_out.norm()

print('=== Advanced Quantization Methods ===')
print(f'\nAWQ (4-bit): relative weight error = {awq_error:.4f}')
print(f'  Top-5 important channels: {importance.topk(5).indices.tolist()}')
print(f'  These channels get higher precision in AWQ')
print(f'\nSmoothQuant: relative output error = {sq_error:.6f}')
print(f'  Smoothly transfers quantization difficulty from activation to weight')
print(f'  Enables W8A8 quantization with minimal accuracy loss')
print(f'\nKey: GPTQ/AWQ for 4-bit weights, SmoothQuant for 8-bit W8A8.')